In [2]:
import numpy as np

In [3]:
np.random.seed(42)

In [4]:
# Simple toy dataset: 4 samples, 2D input, binary labels
X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])  # shape (4, 2)

y = np.array([[0], [1], [1], [0]]) 

In [33]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1-a)
# binary_cross_entropy measures how wrong our predicted probabilities are
def binary_cross_entropy(y_pred, y_true):
    eps = 1e-5
    y_pred_clipped = np.clip(y_pred, eps, 1-eps)
    loss = -(y_true * np.log(y_pred_clipped) + (1-y_true) * np.log(1-y_pred_clipped))
    return np.mean(loss)


In [34]:
# Defining tiny MLP class
class ScratchMLP:
    def __init__(self, input_dim=2, hidden_dim=2, output_dim = 1):

        # weights and biases of first(hidden) layer
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros((1, hidden_dim))

        # weights and biases of second(output) layer
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros((1, output_dim))

        # Placeholders for forward activations
        self.cache = {}

        # Placeholders for gradients
        self.grads = {}

    def forward(self, X):
        # First layer
        z1 = X @ self.W1 + self.b1
        a1 = sigmoid(z1)

        # Second layer
        z2 = a1 @ self.W2 + self.b2
        a2 = sigmoid(z2)

        # Save in cache for backward pass
        self.cache['z1'] = z1
        self.cache['a1'] = a1
        self.cache['z2'] = z2
        self.cache['X'] = X
        self.cache['a2'] = a2

        return a2  #y_pred
    
    def compute_loss(self, X, y_true):
        y_pred = self.forward(X)
        return binary_cross_entropy(y_pred, y_true)
    
    def backward(self, y_true):
    # gradient of the loss wrt w1, b1, w2, b2
        X = self.cache['X']
        a1 = self.cache['a1']
        a2 = self.cache['a2']

        N = X.shape[0]

        # Output layer
        # The gradient loss wrt the preactivation z2
        dz2 = a2 - y_true
        # The weight contribution to error
        dW2 = a1.T @ dz2 / N
        db2 = np.mean(dz2, axis=0, keepdims=True)

        # Hidden layer
        # Error signal flowing backward from output layer to hidden layer
        da1 = dz2 @ self.W2.T
        # Chain rule: multiplying by derivative of activation
        dz1 = da1 * sigmoid_derivative(a1)
    
        dW1 = X.T @ dz1 / N
        db1 = np.mean(dz1, axis=0, keepdims=True)

        # Save gradients
        self.grads['W1'] = dW1
        self.grads['b1'] = db1
        self.grads['W2'] = dW2
        self.grads['b2'] = db2

    

In [38]:
# To do a numerical gradient check, it's easier to treat all parameters as one big vector
def get_params_vector(model):
    params = [
        model.W1.ravel(),
        model.b1.ravel(),
        model.W2.ravel(),
        model.b2.ravel()
    ]
    return np.concatenate(params)

def set_params_vector(model, param_vector):
    idx = 0

    # W1
    W1_size = model.W1.size
    model.W1 = param_vector[idx:idx+W1_size].reshape(model.W1.shape)
    idx += W1_size

    # b1
    b1_size = model.b1.size
    model.b1 = param_vector[idx:idx + b1_size].reshape(model.b1.shape)
    idx += b1_size

    # W2
    W2_size = model.W2.size
    model.W2 = param_vector[idx:idx + W2_size].reshape(model.W2.shape)
    idx += W2_size

    # b2
    b2_size = model.b2.size
    model.b2 = param_vector[idx:idx + b2_size].reshape(model.b2.shape)
    idx += b2_size

def get_grads_vector(model):
    grads = [
        model.grads["W1"].ravel(),
        model.grads["b1"].ravel(),
        model.grads["W2"].ravel(),
        model.grads["b2"].ravel()
    ]

    return np.concatenate(grads)



In [43]:
# Numerical gradient function

def numerical_gradient(model, X, y_true, eps=1e-4):
    original_params = get_params_vector(model)
    num_grads = np.zeros_like(original_params)

    for i in range(len(original_params)):
        # Saving original value
        old_val = original_params[i]

        # w + epsilon for each weight w
        original_params[i] = old_val + eps
        set_params_vector(model, original_params)
        loss_plus = model.compute_loss(X, y_true)

        # w - epsilon
        original_params[i] = old_val - eps
        set_params_vector(model, original_params)
        loss_minus = model.compute_loss(X, y_true)

        # restoring original value
        original_params[i] = old_val

        # numerical derivative
        num_grads[i] = (loss_plus - loss_minus) / (2 * eps)
    
    # restoring original parameters in the model
    set_params_vector(model, original_params)

    return num_grads

In [44]:
# Gradient check

model = ScratchMLP(input_dim=2, hidden_dim=2, output_dim=1)

# Forward + backward pass
y_pred = model.forward(X)
loss = binary_cross_entropy(y_pred, y)
model.backward(y_true=y)

analytical_grads = get_grads_vector(model)

# Numerical gradients
num_grads = numerical_gradient(model, X, y, eps=1e-4)

# Compare gradients
max_diff = np.max(np.abs(analytical_grads - num_grads))

print("Max absolute difference between analytical and numerical gradient is: ", max_diff)


Max absolute difference between analytical and numerical gradient is:  1.4372530943163042e-12
